In [13]:
import tubesml as tml
import pandas as pd
import numpy as np

from source.report import _point_to_proba

from sklearn.metrics import brier_score_loss, mean_squared_error

from sklearn.model_selection import KFold

from sklearn.linear_model import Ridge, LogisticRegression, Lasso
from sklearn.pipeline import Pipeline

import lightgbm as lgb
import xgboost as xgb

import optuna
from optuna.samplers import TPESampler

In [ ]:
df = pd.read_csv('data/processed/men_training.csv')

N_FOLDS = 5
kfolds = KFold(n_splits=N_FOLDS, shuffle=True, random_state=13)

df_train, df_test = tml.make_test(df, test_size=0.2, random_state=34)

DROP = ["target", "target_points", "ID", "DayNum", "Team1", "Team2",
        'T1_Loc', 'T2_Loc',
                "T1_region", "T2_region", "Season", "delta_Loc",
                "Season", "competitive", "competitive_score",
                "delta_def_rating_diff", "delta_impact_diff",
                "T1_def_rating_diff", "T2_def_rating_diff"]

df_train.head()

df_train = df.copy()

## Feats cats

In [3]:
all_feats = [c for c in df_train if c not in DROP]
all_feats

['T1_Ast',
 'T1_Ast_TO_ratio',
 'T1_Ast_TO_ratio_diff',
 'T1_Ast_diff',
 'T1_Away',
 'T1_Blk',
 'T1_Blk_diff',
 'T1_DR',
 'T1_DR_diff',
 'T1_DR_opportunity',
 'T1_DR_opportunity_diff',
 'T1_Eff_FG_perc_diff',
 'T1_FG3_ratio',
 'T1_FG3_ratio_diff',
 'T1_FGA',
 'T1_FGA2',
 'T1_FGA2_diff',
 'T1_FGA3',
 'T1_FGA3_diff',
 'T1_FGA_diff',
 'T1_FGM',
 'T1_FGM2',
 'T1_FGM2_diff',
 'T1_FGM3',
 'T1_FGM3_diff',
 'T1_FGM_diff',
 'T1_FGM_no_ast',
 'T1_FGM_no_ast_diff',
 'T1_FTA',
 'T1_FTA_diff',
 'T1_FTM',
 'T1_FTM_diff',
 'T1_N_wins',
 'T1_OR',
 'T1_OR_diff',
 'T1_OR_opportunity',
 'T1_OR_opportunity_diff',
 'T1_OT_win',
 'T1_PF',
 'T1_PF_diff',
 'T1_Score',
 'T1_Score_diff',
 'T1_Stl',
 'T1_Stl_diff',
 'T1_TO',
 'T1_TO_diff',
 'T1_TO_perposs',
 'T1_TO_perposs_diff',
 'T1_Tot_Reb',
 'T1_Tot_Reb_diff',
 'T1_True_shooting_perc_diff',
 'T1_def_rating',
 'T1_impact',
 'T1_impact_diff',
 'T1_off_rating',
 'T1_off_rating_diff',
 'T1_opp_FGA',
 'T1_opp_FGM',
 'T1_opp_FGM3',
 'T1_opp_FTA',
 'T1_opp_PF',
 'T

In [4]:
all_delta = [c for c in df_train if c not in DROP and "delta" in c]

all_delta

['delta_Ast',
 'delta_Ast_TO_ratio',
 'delta_Ast_TO_ratio_diff',
 'delta_Ast_diff',
 'delta_Away',
 'delta_Blk',
 'delta_Blk_diff',
 'delta_DR',
 'delta_DR_diff',
 'delta_DR_opportunity',
 'delta_DR_opportunity_diff',
 'delta_Eff_FG_perc_diff',
 'delta_FG3_ratio',
 'delta_FG3_ratio_diff',
 'delta_FGA',
 'delta_FGA2',
 'delta_FGA2_diff',
 'delta_FGA3',
 'delta_FGA3_diff',
 'delta_FGA_diff',
 'delta_FGM',
 'delta_FGM2',
 'delta_FGM2_diff',
 'delta_FGM3',
 'delta_FGM3_diff',
 'delta_FGM_diff',
 'delta_FGM_no_ast',
 'delta_FGM_no_ast_diff',
 'delta_FTA',
 'delta_FTA_diff',
 'delta_FTM',
 'delta_FTM_diff',
 'delta_N_wins',
 'delta_OR',
 'delta_OR_diff',
 'delta_OR_opportunity',
 'delta_OR_opportunity_diff',
 'delta_OT_win',
 'delta_PF',
 'delta_PF_diff',
 'delta_Score',
 'delta_Score_diff',
 'delta_Stl',
 'delta_Stl_diff',
 'delta_TO',
 'delta_TO_diff',
 'delta_TO_perposs',
 'delta_TO_perposs_diff',
 'delta_Tot_Reb',
 'delta_Tot_Reb_diff',
 'delta_True_shooting_perc_diff',
 'delta_def_ratin

In [5]:
no_delta = [c for c in df_train if c not in DROP and "delta" not in c]
no_delta

['T1_Ast',
 'T1_Ast_TO_ratio',
 'T1_Ast_TO_ratio_diff',
 'T1_Ast_diff',
 'T1_Away',
 'T1_Blk',
 'T1_Blk_diff',
 'T1_DR',
 'T1_DR_diff',
 'T1_DR_opportunity',
 'T1_DR_opportunity_diff',
 'T1_Eff_FG_perc_diff',
 'T1_FG3_ratio',
 'T1_FG3_ratio_diff',
 'T1_FGA',
 'T1_FGA2',
 'T1_FGA2_diff',
 'T1_FGA3',
 'T1_FGA3_diff',
 'T1_FGA_diff',
 'T1_FGM',
 'T1_FGM2',
 'T1_FGM2_diff',
 'T1_FGM3',
 'T1_FGM3_diff',
 'T1_FGM_diff',
 'T1_FGM_no_ast',
 'T1_FGM_no_ast_diff',
 'T1_FTA',
 'T1_FTA_diff',
 'T1_FTM',
 'T1_FTM_diff',
 'T1_N_wins',
 'T1_OR',
 'T1_OR_diff',
 'T1_OR_opportunity',
 'T1_OR_opportunity_diff',
 'T1_OT_win',
 'T1_PF',
 'T1_PF_diff',
 'T1_Score',
 'T1_Score_diff',
 'T1_Stl',
 'T1_Stl_diff',
 'T1_TO',
 'T1_TO_diff',
 'T1_TO_perposs',
 'T1_TO_perposs_diff',
 'T1_Tot_Reb',
 'T1_Tot_Reb_diff',
 'T1_True_shooting_perc_diff',
 'T1_def_rating',
 'T1_impact',
 'T1_impact_diff',
 'T1_off_rating',
 'T1_off_rating_diff',
 'T1_opp_FGA',
 'T1_opp_FGM',
 'T1_opp_FGM3',
 'T1_opp_FTA',
 'T1_opp_PF',
 'T

In [6]:
seeds = [c for c in df_train if c not in DROP and "Seed" in c] + [c for c in df_train if "quality" in c] + [c for c in df_train if "stage" in c] + [c for c in df_train if "elo" in c]
seeds

['T1_Seed',
 'T2_Seed',
 'delta_Seed',
 'T1_quality',
 'T2_quality',
 'delta_quality',
 'stage_Round1',
 'stage_Round2',
 'stage_Round3',
 'stage_Round4',
 'stage_final',
 'stage_finalfour',
 'stage_impossible',
 'T1_elo',
 'T2_elo',
 'delta_elo']

In [7]:
no_seeds = [c for c in df_train if c not in DROP and "Seed" not in c]
no_seeds

['T1_Ast',
 'T1_Ast_TO_ratio',
 'T1_Ast_TO_ratio_diff',
 'T1_Ast_diff',
 'T1_Away',
 'T1_Blk',
 'T1_Blk_diff',
 'T1_DR',
 'T1_DR_diff',
 'T1_DR_opportunity',
 'T1_DR_opportunity_diff',
 'T1_Eff_FG_perc_diff',
 'T1_FG3_ratio',
 'T1_FG3_ratio_diff',
 'T1_FGA',
 'T1_FGA2',
 'T1_FGA2_diff',
 'T1_FGA3',
 'T1_FGA3_diff',
 'T1_FGA_diff',
 'T1_FGM',
 'T1_FGM2',
 'T1_FGM2_diff',
 'T1_FGM3',
 'T1_FGM3_diff',
 'T1_FGM_diff',
 'T1_FGM_no_ast',
 'T1_FGM_no_ast_diff',
 'T1_FTA',
 'T1_FTA_diff',
 'T1_FTM',
 'T1_FTM_diff',
 'T1_N_wins',
 'T1_OR',
 'T1_OR_diff',
 'T1_OR_opportunity',
 'T1_OR_opportunity_diff',
 'T1_OT_win',
 'T1_PF',
 'T1_PF_diff',
 'T1_Score',
 'T1_Score_diff',
 'T1_Stl',
 'T1_Stl_diff',
 'T1_TO',
 'T1_TO_diff',
 'T1_TO_perposs',
 'T1_TO_perposs_diff',
 'T1_Tot_Reb',
 'T1_Tot_Reb_diff',
 'T1_True_shooting_perc_diff',
 'T1_def_rating',
 'T1_impact',
 'T1_impact_diff',
 'T1_off_rating',
 'T1_off_rating_diff',
 'T1_opp_FGA',
 'T1_opp_FGM',
 'T1_opp_FGM3',
 'T1_opp_FTA',
 'T1_opp_PF',
 'T

In [8]:
feats_dict = {"all_feats": all_feats,
              "all_delta": all_delta, "no_delta": no_delta, "no_seeds": no_seeds, "seeds": seeds}

# Points predictions

## LGBM

In [9]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
        # "agg_func": trial.suggest_categorical("agg_func", ["mean", "median", "std"]),
        # "formula": trial.suggest_categorical("formula", [True, False])
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = lgb.LGBMRegressor(random_state=34, n_jobs=-1, verbose=-1, n_estimators=10000,
                              learning_rate=0.1,
                             colsample_bytree=param["colsample_bytree"],
                             min_child_weight=param['min_child_weight'],
                             reg_lambda=param['reg_lambda'],
                             reg_alpha=param['reg_alpha'],
                             subsample=param['subsample'],
                             num_leaves=param["num_leaves"],
                             max_depth=param['max_depth'],
                             eval_metric="l2")

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    callbacks = [lgb.early_stopping(100, verbose=0)]
    
    fit_params = {"callbacks":callbacks, "eval_metric": "l2"}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [10]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

[I 2026-03-03 21:17:08,430] A new study created in memory with name: no-name-e355630a-44f3-424d-8e8e-5e1683f1a0b6


Number of finished trials: 2000
Best trial: {'max_depth': 121, 'num_leaves': 42, 'reg_lambda': 95.81833535600323, 'reg_alpha': 45.227431904015965, 'colsample_bytree': 0.7614261395398824, 'subsample': 0.4430963675305502, 'min_child_weight': 265.9593792345202, 'feats': 'all_feats', 'clip_val': 45, 'padd': 0.018992776859786894}


In [11]:
0.183152

0.183152

In [12]:
study.trials_dataframe().sort_values('value', ascending=True).head(20)

,number,value,datetime_start,datetime_complete,duration,params_clip_val,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_num_leaves,params_padd,params_reg_alpha,params_reg_lambda,params_subsample,state
1543,1543,0.183068,2026-03-03 23:04:18.502181,2026-03-03 23:05:31.258688,0 days 00:01:12.756507,45,0.761426,all_feats,121,265.959379,42,0.018993,45.227432,95.818335,0.443096,COMPLETE
246,246,0.183069,2026-03-03 21:32:27.483831,2026-03-03 21:33:32.761977,0 days 00:01:05.278146,34,0.808066,all_feats,103,277.186682,36,0.033287,40.590040,89.331520,0.446302,COMPLETE
195,195,0.183109,2026-03-03 21:29:12.996328,2026-03-03 21:30:22.733733,0 days 00:01:09.737405,48,0.761653,all_feats,114,273.967538,19,0.029527,13.863124,99.510298,0.453941,COMPLETE
1172,1172,0.183115,2026-03-03 22:37:41.602633,2026-03-03 22:38:52.359448,0 days 00:01:10.756815,30,0.810492,all_feats,57,266.162753,51,0.015509,57.924689,95.022115,0.615353,COMPLETE
1834,1834,0.183121,2026-03-03 23:25:19.964617,2026-03-03 23:26:29.390853,0 days 00:01:09.426236,34,0.784591,all_feats,67,272.449021,74,0.019222,27.550539,97.853688,0.658698,COMPLETE
546,546,0.183153,2026-03-03 21:53:51.120322,2026-03-03 21:55:02.436135,0 days 00:01:11.315813,34,0.731737,all_feats,99,258.988380,82,0.019774,54.964157,90.645560,0.447653,COMPLETE
1824,1824,0.183155,2026-03-03 23:24:41.115578,2026-03-03 23:25:49.962727,0 days 00:01:08.847149,33,0.780570,all_feats,55,273.435391,37,0.028576,14.996637,92.288403,0.444500,COMPLETE
1327,1327,0.183194,2026-03-03 22:49:00.064065,2026-03-03 22:50:06.092078,0 days 00:01:06.028013,35,0.787752,all_feats,68,284.079482,35,0.019364,13.722486,91.518803,0.831087,COMPLETE
184,184,0.183210,2026-03-03 21:28:21.335115,2026-03-03 21:29:32.582474,0 days 00:01:11.247359,47,0.723271,all_feats,80,275.651214,35,0.017445,17.917813,91.843270,0.432668,COMPLETE
621,621,0.183228,2026-03-03 21:58:31.585828,2026-03-03 21:59:28.048529,0 days 00:00:56.462701,34,0.766845,all_feats,119,285.507623,40,0.015245,17.130757,81.144811,0.634465,COMPLETE


## XGBoost

In [16]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
        # "agg_func": trial.suggest_categorical("agg_func", ["mean", "median", "std"]),
        # "formula": trial.suggest_categorical("formula", [True, False])
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = xgb.XGBRegressor(random_state=34, n_jobs=-1, n_estimators=10000,
                              learning_rate=0.1,
                             subsample=param["subsample"],
                            colsample_bytree=param["colsample_bytree"],
                            reg_alpha=param["reg_alpha"],
                            reg_lambda=param["reg_lambda"],
                            max_depth=param["max_depth"],
                            colsample_bylevel=param["colsample_bylevel"],
                            early_stopping_rounds=100,
                             eval_metric=mean_squared_error)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    
    fit_params = {'verbose': False}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [17]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 2
Best trial: {'max_depth': 32, 'reg_lambda': 34.20172928288744, 'reg_alpha': 70.68383323593484, 'colsample_bytree': 0.4243991097547476, 'colsample_bylevel': 0.5081487983426731, 'subsample': 0.5971648284184927, 'min_child_weight': 225.31095442959224, 'feats': 'seeds', 'clip_val': 38, 'padd': 0.003642288794427989}


,number,value,datetime_start,datetime_complete,duration,params_clip_val,params_colsample_bylevel,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_padd,params_reg_alpha,params_reg_lambda,params_subsample,state
1,1,0.188673,2026-03-06 22:28:43.006064,2026-03-06 22:29:32.327488,0 days 00:00:49.321424,38,0.508149,0.424399,seeds,32,225.310954,0.003642,70.683833,34.201729,0.597165,COMPLETE
0,0,0.192275,2026-03-06 22:28:43.004802,2026-03-06 22:30:24.252917,0 days 00:01:41.248115,36,0.686327,0.953021,no_delta,172,204.481176,0.026944,82.100972,93.073478,0.709368,COMPLETE


## Ridge

In [ ]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "alpha": trial.suggest_float("alpha", 0.1, 200),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = Ridge(random_state=34, alpha=param["alpha"], max_iter=10000)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [ ]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Ill-conditioned matrix (rcond=1.40179e-25): result may not be accurate.
Ill-conditioned matrix (rcond=6.09801e-26): result may not be accurate.
Ill-conditioned matrix (rcond=1.34623e-25): result may not be accurate.
Ill-conditioned matrix (rcond=1.34342e-25): result may not be accurate.
Ill-conditioned matrix (rcond=1.34623e-25): result may not be accurate.
Ill-conditioned matrix (rcond=1.34913e-25): result may not be accurate.
Ill-conditioned matrix (rcond=1.34601e-25): result may not be accurate.
Ill-conditioned matrix (rcond=1.40525e-25): result may not be accurate.
Ill-conditioned matrix (rcond=6.00164e-26): result may not be accurate.
Ill-conditioned matrix (rcond=1.65388e-25): result may not be accurate.


Number of finished trials: 2
Best trial: {'alpha': 198.32672439433063, 'feats': 'all_feats', 'clip_val': 42, 'padd': 0.02535014200662916}


,number,value,datetime_start,datetime_complete,duration,params_alpha,params_clip_val,params_feats,params_padd,state
0,0,0.186186,2026-03-06 22:30:45.028453,2026-03-06 22:30:48.700930,0 days 00:00:03.672477,198.326724,42,all_feats,0.025350,COMPLETE
1,1,0.186413,2026-03-06 22:30:45.028835,2026-03-06 22:30:47.949595,0 days 00:00:02.920760,153.933399,33,no_delta,0.032429,COMPLETE


## Lasso

In [22]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "alpha": trial.suggest_float("alpha", 0.1, 200),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = Lasso(random_state=34, alpha=param["alpha"], max_iter=10000)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [ ]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.517e+04, tolerance: 5.376e+01
Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.524e+04, tolerance: 5.376e+01
Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.903e+04, tolerance: 5.424e+01


Number of finished trials: 2
Best trial: {'alpha': 104.5419036477974, 'feats': 'no_seeds', 'clip_val': 40, 'padd': 0.007407736147311834}


Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.911e+04, tolerance: 5.424e+01


,number,value,datetime_start,datetime_complete,duration,params_alpha,params_clip_val,params_feats,params_padd,state
1,1,0.207208,2026-03-06 22:31:42.223584,2026-03-06 22:31:52.462896,0 days 00:00:10.239312,104.541904,40,no_seeds,0.007408,COMPLETE
0,0,0.207210,2026-03-06 22:31:42.222556,2026-03-06 22:31:53.033148,0 days 00:00:10.810592,106.934260,45,all_feats,0.023130,COMPLETE


# Probability Predictions


## LGBM

In [ ]:
def objective(trial, data=df_train, target=df_train["target"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = lgb.LGBMClassifier(random_state=34, n_jobs=-1, verbose=-1, n_estimators=10000,
                               learning_rate=0.1,
                             colsample_bytree=param["colsample_bytree"],
                             min_child_weight=param['min_child_weight'],
                             reg_lambda=param['reg_lambda'],
                             reg_alpha=param['reg_alpha'],
                             subsample=param['subsample'],
                             num_leaves=param["num_leaves"],
                             max_depth=param['max_depth'],
                             eval_metric="auc")

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    callbacks = [lgb.early_stopping(100, verbose=0)]
    
    fit_params = {"callbacks":callbacks, "eval_metric": "auc"}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True, predict_proba=True)
    oof, _ = cvscore.score()

    score = brier_score_loss(target, y_prob=oof)
    
    return score

In [ ]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 1000
Best trial: {'max_depth': 92, 'num_leaves': 65, 'reg_lambda': 64.41003814135934, 'reg_alpha': 4.539626259888374, 'colsample_bytree': 0.38027269932810837, 'subsample': 0.9477615775063852, 'min_child_weight': 5.909604890995187, 'feats': 'all_feats'}


## XGBoost

In [26]:
def objective(trial, data=df_train, target=df_train["target"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = xgb.XGBClassifier(random_state=34, n_jobs=-1, n_estimators=10000,
                              learning_rate=0.1,
                             subsample=param["subsample"],
                            colsample_bytree=param["colsample_bytree"],
                            reg_alpha=param["reg_alpha"],
                            reg_lambda=param["reg_lambda"],
                            max_depth=param["max_depth"],
                            colsample_bylevel=param["colsample_bylevel"],
                            early_stopping_rounds=100,
                             eval_metric="auc")

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    
    fit_params = {"verbose": False}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True, predict_proba=True)
    oof, _ = cvscore.score()

    score = brier_score_loss(target, y_prob=oof)
    
    return score

In [ ]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 2
Best trial: {'max_depth': 162, 'reg_lambda': 70.17506837767813, 'reg_alpha': 4.132375550784136, 'colsample_bytree': 0.387548608778664, 'colsample_bylevel': 0.45041705184948105, 'subsample': 0.43861729539134453, 'min_child_weight': 175.7727531303093, 'feats': 'all_delta'}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bylevel,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_reg_alpha,params_reg_lambda,params_subsample,state
0,0,0.190540,2026-03-06 22:35:52.751439,2026-03-06 22:36:33.714438,0 days 00:00:40.962999,0.450417,0.387549,all_delta,162,175.772753,4.132376,70.175068,0.438617,COMPLETE
1,1,0.222053,2026-03-06 22:35:52.752763,2026-03-06 22:36:23.141505,0 days 00:00:30.388742,0.985694,0.598250,no_delta,84,218.797453,69.001029,72.904040,0.503682,COMPLETE


## LogisticRegression

In [28]:
def objective(trial, data=df_train, target=df_train["target"]):
    param = {
        'C': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = LogisticRegression(C=param["C"], random_state=34, max_iter=10000)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, predict_proba=True)
    oof, _ = cvscore.score()

    score = brier_score_loss(target, y_prob=oof)
    
    return score

In [ ]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 2
Best trial: {'reg_lambda': 65.39341607259794, 'reg_alpha': 73.22811168744762, 'colsample_bytree': 0.6810585583938084, 'colsample_bylevel': 0.5791759968305761, 'subsample': 0.8474463482350314, 'min_child_weight': 151.92959129959337, 'feats': 'no_delta'}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bylevel,params_colsample_bytree,params_feats,params_min_child_weight,params_reg_alpha,params_reg_lambda,params_subsample,state
0,0,0.250125,2026-03-06 22:38:26.986068,2026-03-06 22:38:30.482272,0 days 00:00:03.496204,0.579176,0.681059,no_delta,151.929591,73.228112,65.393416,0.847446,COMPLETE
1,1,0.250125,2026-03-06 22:38:26.987198,2026-03-06 22:38:31.432114,0 days 00:00:04.444916,0.508328,0.954967,all_feats,177.977232,44.335861,56.510228,0.423041,COMPLETE
